<a href="https://colab.research.google.com/github/Radhakuchekar/Preparation/blob/pyspark/HashtagAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


```
Question:
  Hashtag Analysis from Social Media Posts
  You are given a dataset containing social media posts, stored in a text file. Each line represents a single post with the following format:
  user_id, timestamp, post_text
  101, 2025-02-28T10:30:00Z, "Loving the new AI tools! #AI #MachineLearning"
  102, 2025-02-28T10:32:15Z, "Big data processing is amazing! #BigData #AI"
  103, 2025-02-28T10:35:45Z, "Spark RDDs are powerful. #Spark #BigData"

Tasks:

    *   The top 5 most used hashtags.
    *   The number of unique users who used each hashtag

Solution Observations:
    *  Timestamp : unix_timesptamp() works on timestamp data only if string passed returns null instead use to_timestamp(col(""), "timestamp format")
    *   Avoid frequent show and collect statements to avoid unnecessary actions. used here for learning purposes

# Approach Efficiency

  Feature	          DataFrame API (Approach 1)	    RDD API (Approach 2)
  Execution Engine     Catalyst & Tungsten (Optimized)   Manual Execution
  Ease of Use	      SQL-like, Readable	            More complex, requires manual tuning
  Scalability	      Highly optimized for Big Data	 Can cause performance issues at scale
  Optimization	     Automatic via Catalyst	        Requires manual partitioning & caching
  Shuffle Overhead	 Lower	                         Higher (due to map-reduce style ops)
  Memory Usage	     Lower (Columnar storage)	      Higher (Row-based processing)
  Query Performance	Faster due to optimization	    Slower due to shuffling & collect()

Conclusion: Which Approach is More Efficient?
  For large-scale distributed data processing, the DataFrame API (Approach 1) is significantly more efficient.
  Reason: It leverages Spark SQL optimizations, uses a columnar format, and minimizes data shuffling.
  When to use RDDs? Only when low-level control or custom optimizations are needed (e.g., handling unstructured data, using custom partitioning).
```



In [84]:

!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
from pyspark.sql.functions import *
sc = spark.sparkContext
spark

In [82]:
columns = ["user_id", "timestamp", "post_text"]

data = [
  [101, "2025-02-28T10:30:00Z", "Loving the new AI tools! #AI #MachineLearning"],
  [102, "2025-02-28T10:32:15Z", "Big data processing is amazing! #BigData #AI"],
  [103, "2025-02-28T10:35:45Z", "Spark RDDs are powerful. #Spark #BigData #AI #PySpark #Colab"]
]

# Data Frames


In [63]:
posts_data = spark.createDataFrame(data, columns)
posts_data.show(truncate= False)

+-------+--------------------+------------------------------------------------------------+
|user_id|timestamp           |post_text                                                   |
+-------+--------------------+------------------------------------------------------------+
|101    |2025-02-28T10:30:00Z|Loving the new AI tools! #AI #MachineLearning               |
|102    |2025-02-28T10:32:15Z|Big data processing is amazing! #BigData #AI                |
|103    |2025-02-28T10:35:45Z|Spark RDDs are powerful. #Spark #BigData #AI #PySpark #Colab|
+-------+--------------------+------------------------------------------------------------+



In [64]:
#timestamp column from string to timestamp time
posts_data.schema

StructType([StructField('user_id', LongType(), True), StructField('timestamp', StringType(), True), StructField('post_text', StringType(), True)])

In [65]:
posts_data= posts_data.withColumn("timestamp", to_timestamp("timestamp", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
posts_data.show(truncate = False)


+-------+-------------------+------------------------------------------------------------+
|user_id|timestamp          |post_text                                                   |
+-------+-------------------+------------------------------------------------------------+
|101    |2025-02-28 10:30:00|Loving the new AI tools! #AI #MachineLearning               |
|102    |2025-02-28 10:32:15|Big data processing is amazing! #BigData #AI                |
|103    |2025-02-28 10:35:45|Spark RDDs are powerful. #Spark #BigData #AI #PySpark #Colab|
+-------+-------------------+------------------------------------------------------------+



In [66]:
# select only required column
post_text = posts_data.select(["user_id", "post_text"])

In [67]:
hashtags = (post_text.withColumn("words", split(col("post_text"), " "))
            .withColumn("hashtags", expr("filter(words, w -> w like '#%')"))
            .drop("words", "post_text"))
hashtags.show(truncate = False)

+-------+-----------------------------------------+
|user_id|hashtags                                 |
+-------+-----------------------------------------+
|101    |[#AI, #MachineLearning]                  |
|102    |[#BigData, #AI]                          |
|103    |[#Spark, #BigData, #AI, #PySpark, #Colab]|
+-------+-----------------------------------------+



In [68]:
hashtags = hashtags.withColumn("hashtags", explode("hashtags"))

In [72]:
# The top 5 most used hashtags.
top_five_hashtags = hashtags.groupBy("hashtags").agg(count(col("user_id")).alias("hashtag_count")).orderBy(desc("hashtag_count")).limit(5)
top_five_hashtags.show()

+----------------+-------------+
|        hashtags|hashtag_count|
+----------------+-------------+
|             #AI|            3|
|        #BigData|            2|
|#MachineLearning|            1|
|          #Colab|            1|
|          #Spark|            1|
+----------------+-------------+



In [81]:
# The number of unique users who used each hashtag
hashtags.join(top_five_hashtags, "hashtags").select(["hashtags", "user_id"]).orderBy("hashtags").show(truncate = False)

+----------------+-------+
|hashtags        |user_id|
+----------------+-------+
|#AI             |102    |
|#AI             |101    |
|#AI             |103    |
|#BigData        |102    |
|#BigData        |103    |
|#Colab          |103    |
|#MachineLearning|101    |
|#Spark          |103    |
+----------------+-------+



# RDDs

In [83]:
data

[[101,
  '2025-02-28T10:30:00Z',
  'Loving the new AI tools! #AI #MachineLearning'],
 [102, '2025-02-28T10:32:15Z', 'Big data processing is amazing! #BigData #AI'],
 [103,
  '2025-02-28T10:35:45Z',
  'Spark RDDs are powerful. #Spark #BigData #AI #PySpark #Colab']]

In [127]:
posts_rdd= sc.parallelize(data)
#Extract (user_id, hashtag) pairs using flatMap
hashtags_rdd = posts_rdd.flatMap(lambda row: [(row[0], word) for word in row[2].split() if word.startswith("#")])
print(hashtags_rdd.collect())

[(101, '#AI'), (101, '#MachineLearning'), (102, '#BigData'), (102, '#AI'), (103, '#Spark'), (103, '#BigData'), (103, '#AI'), (103, '#PySpark'), (103, '#Colab')]


In [128]:
#Count total occurrences of each hashtag
hashtag_counts = hashtags_rdd.map(lambda x: (x[1], 1)) \
                             .reduceByKey(lambda a, b: a + b)
print(hashtag_counts.collect())

[('#PySpark', 1), ('#Colab', 1), ('#AI', 3), ('#MachineLearning', 1), ('#BigData', 2), ('#Spark', 1)]


In [129]:
# Get the top 5 most used hashtags (Optimized using top())
top_five_hashtags = hashtag_counts.top(5, key=lambda x: x[1])

print(top_five_hashtags)

[('#AI', 3), ('#BigData', 2), ('#PySpark', 1), ('#Colab', 1), ('#MachineLearning', 1)]


In [131]:
#Extract users who used top hashtags efficiently
top_five_hashtags_set = {hashtag for hashtag, _ in top_five_hashtags}
top_users_rdd = hashtags_rdd.filter(lambda x: x[1] in top_five_hashtags_set).distinct()
print(top_users_rdd.collect())

[(101, '#AI'), (101, '#MachineLearning'), (103, '#BigData'), (103, '#AI'), (102, '#BigData'), (102, '#AI'), (103, '#PySpark'), (103, '#Colab')]


```
# Approach Efficiency

Feature	          DataFrame API (Approach 1)	    RDD API (Approach 2)
Execution Engine     Catalyst & Tungsten (Optimized)   Manual Execution
Ease of Use	      SQL-like, Readable	            More complex, requires manual tuning
Scalability	      Highly optimized for Big Data	 Can cause performance issues at scale
Optimization	     Automatic via Catalyst	        Requires manual partitioning & caching
Shuffle Overhead	 Lower	                         Higher (due to map-reduce style ops)
Memory Usage	     Lower (Columnar storage)	      Higher (Row-based processing)
Query Performance	Faster due to optimization	    Slower due to shuffling & collect()



Conclusion: Which Approach is More Efficient?

For large-scale distributed data processing, the DataFrame API (Approach 1) is significantly more efficient.

Reason: It leverages Spark SQL optimizations, uses a columnar format, and minimizes data shuffling.
When to use RDDs? Only when low-level control or custom optimizations are needed (e.g., handling unstructured data, using custom partitioning).
```
